In [ ]:
!pip install --quiet --upgrade transformers datasets accelerate seqeval scikit-learn

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 86.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.5/527.5 kB 36.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 80.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 19.5 MB/s eta 0:00:00


In [ ]:
import os
import re
import json
import random
from collections import defaultdict
from dataclasses import dataclass
from pathlib import Path
from typing import List, Dict, Tuple, Any

import numpy as np
import torch
from torch.utils.data import Dataset

from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    Trainer,
    TrainingArguments,
    DataCollatorForTokenClassification,
    set_seed,
)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
os.path.exists("/content/drive/MyDrive/biomedical_competitions_2026")

True

In [ ]:
DATA_ROOT = Path("/content/drive/MyDrive/biomedical_competitions_2026/multiclinai/data/MultiClinNER/MultiClinNER-en/MultiClinNER-en-train/")
os.listdir(DATA_ROOT)

OUTPUT_DIR = Path("/content/drive/MyDrive/Colab Notebooks/clinicalbert_en")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

### Configuration

In [ ]:
LABELS = ["O", "B-DISEASE", "I-DISEASE", "B-SYMPTOM", "I-SYMPTOM", "B-PROCEDURE", "I-PROCEDURE"]
LABEL2ID = {x: i for i, x in enumerate(LABELS)}
ID2LABEL = {i: x for x, i in LABEL2ID.items()}

ENTITY_TYPES = {
    "disease": "DISEASE",
    "symptom": "SYMPTOM",
    "procedure": "PROCEDURE",
}

MODEL_NAME = "emilyalsentzer/Bio_ClinicalBERT"
# MODEL_NAME = "medicalai/ClinicalBERT"

MAX_LENGTH = 256
STRIDE = 64
LEARNING_RATE = 2e-5
TRAIN_BATCH_SIZE = 16
EVAL_BATCH_SIZE = 16
EPOCHS = 5
VALID_RATIO = 0.1
SEED = 42

set_seed(SEED)

torch.backends.cuda.matmul.allow_tf32 = True

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

Device: cuda


### BRAT parsing and data loading

In [ ]:
def parse_ann_file(ann_path: Path) -> List[Dict[str, Any]]:
    spans = []
    if not ann_path.exists():
        return spans

    with ann_path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line or not line.startswith("T"):
                continue

            parts = line.split("\t")
            if len(parts) < 3:
                continue

            tid = parts[0]
            tag_and_offsets = parts[1]
            text = parts[2]

            first_space = tag_and_offsets.find(" ")
            if first_space == -1:
                continue

            label = tag_and_offsets[:first_space]
            offsets_str = tag_and_offsets[first_space + 1:]

            for seg in offsets_str.split(";"):
                seg = seg.strip()
                m = re.match(r"^(\d+)\s+(\d+)$", seg)
                if not m:
                    continue
                start, end = int(m.group(1)), int(m.group(2))
                spans.append({
                    "id": tid,
                    "label": label,
                    "start": start,
                    "end": end,
                    "text": text,
                })
    return spans


def load_multiclinner_documents(train_root: Path):
    """
    Merge disease/symptom/procedure annotations into one document per doc_id.
    """
    doc_map = {}

    for subfolder_name, entity_label in ENTITY_TYPES.items():
        base = train_root / f"MultiClinNER-en-train-{subfolder_name}"
        txt_dir = base / "txt"
        ann_dir = base / "ann"

        if not txt_dir.exists():
            raise FileNotFoundError(f"Missing folder: {txt_dir}")

        for txt_path in sorted(txt_dir.glob("*.txt")):
            doc_id = txt_path.stem
            ann_path = ann_dir / f"{doc_id}.ann"
            text = txt_path.read_text(encoding="utf-8")
            spans = parse_ann_file(ann_path)

            # keep only this entity type from this subfolder
            spans = [s for s in spans if s["label"].upper() == entity_label]

            if doc_id not in doc_map:
                doc_map[doc_id] = {
                    "doc_id": doc_id,
                    "text": text,
                    "spans": [],
                }
            else:
                # safety check: same doc_id should have same text in all folders
                if doc_map[doc_id]["text"] != text:
                    raise ValueError(f"Text mismatch for doc_id={doc_id}")

            doc_map[doc_id]["spans"].extend(spans)

    docs = list(doc_map.values())

    # sort spans by start/end/label
    for doc in docs:
        doc["spans"] = sorted(
            doc["spans"],
            key=lambda x: (x["start"], x["end"], x["label"])
        )

    return docs

In [ ]:
docs = load_multiclinner_documents(DATA_ROOT)

print("Documents:", len(docs))
print("doc_id:", docs[0]["doc_id"])
print("num_spans:", len(docs[0]["spans"]))
print("first_spans:", docs[0]["spans"][:10])

Documents: 3774
doc_id: MultiClinNER-en-train-disease-0001
num_spans: 4
first_spans: [{'id': 'T1', 'label': 'DISEASE', 'start': 60, 'end': 68, 'text': 'diabetes'}, {'id': 'T5', 'label': 'DISEASE', 'start': 91, 'end': 129, 'text': 'moderate acalculous acute pancreatitis'}, {'id': 'T2', 'label': 'DISEASE', 'start': 486, 'end': 493, 'text': 'syncope'}, {'id': 'T3', 'label': 'DISEASE', 'start': 643, 'end': 677, 'text': 'haemorrhagic pancreatic pseudocyst'}]


Inspect a few merged documents

In [ ]:
for i in range(min(3, len(docs))):
    print("=" * 80)
    print("doc_id:", docs[i]["doc_id"])
    print("num_spans:", len(docs[i]["spans"]))
    print("first_spans:", docs[i]["spans"][:10])

doc_id: MultiClinNER-en-train-disease-0001
num_spans: 4
first_spans: [{'id': 'T1', 'label': 'DISEASE', 'start': 60, 'end': 68, 'text': 'diabetes'}, {'id': 'T5', 'label': 'DISEASE', 'start': 91, 'end': 129, 'text': 'moderate acalculous acute pancreatitis'}, {'id': 'T2', 'label': 'DISEASE', 'start': 486, 'end': 493, 'text': 'syncope'}, {'id': 'T3', 'label': 'DISEASE', 'start': 643, 'end': 677, 'text': 'haemorrhagic pancreatic pseudocyst'}]
doc_id: MultiClinNER-en-train-disease-0002
num_spans: 4
first_spans: [{'id': 'T1', 'label': 'DISEASE', 'start': 63, 'end': 84, 'text': 'arterial hypertension'}, {'id': 'T2', 'label': 'DISEASE', 'start': 89, 'end': 112, 'text': 'ischaemic heart disease'}, {'id': 'T3', 'label': 'DISEASE', 'start': 178, 'end': 185, 'text': 'anaemia'}, {'id': 'T4', 'label': 'DISEASE', 'start': 357, 'end': 413, 'text': 'intestinal perforation at the level of the sigmoid colon'}]
doc_id: MultiClinNER-en-train-disease-0003
num_spans: 9
first_spans: [{'id': 'T1', 'label': 'DIS

Inspect one full example manually

In [ ]:
doc = docs[0]
print("doc_id:", doc["doc_id"])
print("\nTEXT PREVIEW:\n")
print(doc["text"][:1500])
print("\nSPANS:\n")
print(doc["spans"][:20])

doc_id: MultiClinNER-en-train-disease-0001

TEXT PREVIEW:

We present the case of a 43-year-old man, with a history of diabetes, who was admitted for moderate acalculous acute pancreatitis. Two weeks later, he reported acute pain in the left hemiabdomen and nausea, with no other symptoms. On examination, a large occupation of the left hemiabdomen was palpated without peritoneal irritation. Laboratory tests: elevated triglycerides and lipase, other biochemistry, haemogram and coagulation normal. During the examination, the patient suffered syncope and his haematocrit dropped from 40 to 21%, requiring transfusion of 6 red blood cell concentrates, and he remained stable. Imaging tests revealed a haemorrhagic pancreatic pseudocyst measuring 17 x 27 cm in diameter. The patient was admitted to the ICU where he remained stable. One week later, the patient was scheduled for surgery with a cysto-jejunostomy. The postoperative period was uneventful and the patient was discharged one week after t

### Train / Validation split

In [ ]:
def split_docs(docs, valid_ratio=0.1, seed=42):
    rng = random.Random(seed)
    docs = docs.copy()
    rng.shuffle(docs)
    n_valid = max(1, int(len(docs) * valid_ratio))
    valid_docs = docs[:n_valid]
    train_docs = docs[n_valid:]
    return train_docs, valid_docs

train_docs, valid_docs = split_docs(docs, valid_ratio=VALID_RATIO, seed=SEED)

print("Train docs:", len(train_docs))
print("Valid docs:", len(valid_docs))

Train docs: 3397
Valid docs: 377


### Token-label alignment and dataset

In [ ]:
def char_label_for_offset(token_start: int, token_end: int, spans: List[Dict[str, Any]]) -> str:
    for span in spans:
        s, e, ent = span["start"], span["end"], span["label"].upper()

        # require token to be fully inside the entity span
        if token_start >= s and token_end <= e:
            if token_start == s:
                return f"B-{ent}"
            else:
                return f"I-{ent}"

    return "O"


@dataclass
class ChunkRecord:
    doc_id: str
    chunk_id: int
    text: str
    input_ids: List[int]
    attention_mask: List[int]
    labels: List[int]
    offsets: List[Tuple[int, int]]


class MultiClinNERDataset(Dataset):
    def __init__(self, docs, tokenizer, max_length=256, stride=64):
        self.records = []

        for doc in docs:
            text = doc["text"]
            spans = doc["spans"]

            enc = tokenizer(
                text,
                return_offsets_mapping=True,
                truncation=True,
                max_length=max_length,
                stride=stride,
                return_overflowing_tokens=True,
                padding=False,
            )

            for chunk_idx in range(len(enc["input_ids"])):
                input_ids = enc["input_ids"][chunk_idx]
                attention_mask = enc["attention_mask"][chunk_idx]
                offsets = enc["offset_mapping"][chunk_idx]

                labels = []
                for token_id, (start, end) in zip(input_ids, offsets):
                    if start == 0 and end == 0:
                        labels.append(-100)
                        continue

                    lbl = char_label_for_offset(start, end, spans)
                    labels.append(LABEL2ID[lbl])

                self.records.append(
                    ChunkRecord(
                        doc_id=doc["doc_id"],
                        chunk_id=chunk_idx,
                        text=text,
                        input_ids=input_ids,
                        attention_mask=attention_mask,
                        labels=labels,
                        offsets=offsets,
                    )
                )

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        r = self.records[idx]
        return {
            "input_ids": r.input_ids,
            "attention_mask": r.attention_mask,
            "labels": r.labels,
        }

### Tokenizer and datasets

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)

train_ds = MultiClinNERDataset(
    train_docs,
    tokenizer,
    max_length=MAX_LENGTH,
    stride=STRIDE
)

valid_ds = MultiClinNERDataset(
    valid_docs,
    tokenizer,
    max_length=MAX_LENGTH,
    stride=STRIDE
)

print("Train chunks:", len(train_ds))
print("Valid chunks:", len(valid_ds))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

Train chunks: 18076
Valid chunks: 2151


### Label distribution check

In [ ]:
from collections import Counter

counter = Counter()
for rec in train_ds.records:
    for lab in rec.labels:
        if lab != -100:
            counter[ID2LABEL[lab]] += 1

print(counter)

Counter({'O': 3703937, 'I-SYMPTOM': 177786, 'I-DISEASE': 161584, 'I-PROCEDURE': 137084, 'B-SYMPTOM': 31339, 'B-PROCEDURE': 30042, 'B-DISEASE': 27075})


### Class-weight check

In [ ]:
from collections import Counter
import torch

label_counter = Counter()
for rec in train_ds.records:
    for lab in rec.labels:
        if lab != -100:
            label_counter[lab] += 1

print({ID2LABEL[k]: v for k, v in sorted(label_counter.items())})

total = sum(label_counter.values())
num_classes = len(LABELS)

class_weights = torch.ones(num_classes, dtype=torch.float)

for lab_id in range(num_classes):
    count = label_counter.get(lab_id, 1)
    class_weights[lab_id] = total / (num_classes * count)

# do not overweight O
class_weights[LABEL2ID["O"]] = min(class_weights[LABEL2ID["O"]], 1.0)

print("Class weights:")
for i, w in enumerate(class_weights.tolist()):
    print(ID2LABEL[i], round(w, 4))

{'O': 3703937, 'B-DISEASE': 27075, 'I-DISEASE': 161584, 'B-SYMPTOM': 31339, 'I-SYMPTOM': 177786, 'B-PROCEDURE': 30042, 'I-PROCEDURE': 137084}
Class weights:
O 0.1646
B-DISEASE 22.5239
I-DISEASE 3.7741
B-SYMPTOM 19.4593
I-SYMPTOM 3.4302
B-PROCEDURE 20.2994
I-PROCEDURE 4.4486


##### Optional token-level sanity check

In [ ]:
sample = train_ds.records[0]

tokens = tokenizer.convert_ids_to_tokens(sample.input_ids)
labels = [ID2LABEL[x] if x != -100 else "IGN" for x in sample.labels]

for tok, lab in list(zip(tokens, labels))[:80]:
    print(f"{tok:20s} {lab}")

[CLS]                IGN
a                    O
46                   O
-                    O
year                 O
-                    O
old                  O
woman                O
with                 O
d                    B-SYMPTOM
##ys                 I-SYMPTOM
##pe                 I-SYMPTOM
##ps                 I-SYMPTOM
##ia                 I-SYMPTOM
of                   O
several              O
years                O
'                    O
duration             O
that                 O
did                  O
not                  O
improve              O
with                 O
medical              O
-                    O
diet                 O
##ary                O
treatment            O
.                    O
end                  O
##os                 O
##copy               O
revealed             O
a                    O
sub                  O
##mu                 O
##cos                O
##al                 O
les                  O
##ion                O
1              

### Weighted trainer definition

In [ ]:
import torch.nn as nn
from transformers import Trainer

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits

        loss_fct = nn.CrossEntropyLoss(
            weight=class_weights.to(logits.device),
            ignore_index=-100
        )
        loss = loss_fct(logits.view(-1, model.config.num_labels), labels.view(-1))

        return (loss, outputs) if return_outputs else loss

### Model and Trainer

In [ ]:
model = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(LABELS),
    id2label=ID2LABEL,
    label2id=LABEL2ID,
)

data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR),
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    num_train_epochs=EPOCHS,
    weight_decay=0.01,
    logging_steps=50,
    save_strategy="epoch",
    eval_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    report_to="none",
    fp16=torch.cuda.is_available(),
    save_total_limit=2,
)

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=valid_ds,
    data_collator=data_collator,
    processing_class=tokenizer,
)

pytorch_model.bin:   0%|          | 0.00/436M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: emilyalsentzer/Bio_ClinicalBERT
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you

### Train

In [ ]:
trainer.train()
trainer.save_model(str(OUTPUT_DIR))
tokenizer.save_pretrained(str(OUTPUT_DIR))

model.safetensors:   0%|          | 0.00/436M [00:00<?, ?B/s]

Epoch,Training Loss,Validation Loss
1,0.544538,0.529073
2,0.487241,0.518929
3,0.433624,0.532234
4,0.401067,0.550222
5,0.385133,0.571400


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('/content/drive/MyDrive/Colab Notebooks/clinicalbert_en/tokenizer_config.json',
 '/content/drive/MyDrive/Colab Notebooks/clinicalbert_en/tokenizer.json')

### Span decoding and exact-span evaluation

In [ ]:
def aggregate_token_predictions(model, tokenizer, text, max_length=256, stride=64, device="cpu"):
    model.eval()

    enc = tokenizer(
        text,
        return_offsets_mapping=True,
        truncation=True,
        max_length=max_length,
        stride=stride,
        return_overflowing_tokens=True,
        padding=False,
    )

    token_votes = defaultdict(list)

    with torch.no_grad():
        for chunk_idx in range(len(enc["input_ids"])):
            input_ids = torch.tensor([enc["input_ids"][chunk_idx]], device=device)
            attention_mask = torch.tensor([enc["attention_mask"][chunk_idx]], device=device)
            offsets = enc["offset_mapping"][chunk_idx]

            out = model(input_ids=input_ids, attention_mask=attention_mask)
            pred_ids = out.logits.argmax(-1).squeeze(0).cpu().tolist()

            for pred_id, (start, end) in zip(pred_ids, offsets):
                if start == 0 and end == 0:
                    continue
                token_votes[(start, end)].append(pred_id)

    merged_tokens = []
    for (start, end), votes in token_votes.items():
        voted_label_id = Counter(votes).most_common(1)[0][0]
        merged_tokens.append((start, end, voted_label_id))

    merged_tokens.sort(key=lambda x: (x[0], x[1]))
    return merged_tokens


def decode_spans_from_aggregated_tokens(merged_tokens):
    spans = []

    current_start = None
    current_end = None
    current_type = None

    for start, end, pred_id in merged_tokens:
        label = ID2LABEL[int(pred_id)]

        if label == "O":
            if current_type is not None:
                spans.append((current_start, current_end, current_type))
                current_start, current_end, current_type = None, None, None
            continue

        prefix, ent_type = label.split("-", 1)

        if prefix == "B":
            if current_type is not None:
                spans.append((current_start, current_end, current_type))
            current_start, current_end, current_type = start, end, ent_type

        elif prefix == "I":
            if current_type == ent_type and current_end is not None and start <= current_end:
                current_end = max(current_end, end)
            else:
                if current_type is not None:
                    spans.append((current_start, current_end, current_type))
                current_start, current_end, current_type = start, end, ent_type

    if current_type is not None:
        spans.append((current_start, current_end, current_type))

    return spans


def clean_predicted_spans(text, spans):
    cleaned = []

    for start, end, label in spans:
        if start >= end:
            continue

        while start < end and text[start].isspace():
            start += 1
        while end > start and text[end - 1].isspace():
            end -= 1

        if start >= end:
            continue

        cleaned.append((start, end, label))

    cleaned = sorted(set(cleaned), key=lambda x: (x[0], x[1], x[2]))
    return cleaned


def merge_adjacent_same_type_spans(text, spans, max_gap=1):
    """
    Merge adjacent predicted spans of the same type if the text between them
    is only whitespace or a very small gap.

    max_gap=1 allows:
      "neurological examination"
      "CT scan"
    but avoids merging distant spans.
    """
    if not spans:
        return spans

    spans = sorted(spans, key=lambda x: (x[0], x[1], x[2]))
    merged = [spans[0]]

    for start, end, label in spans[1:]:
        prev_start, prev_end, prev_label = merged[-1]

        gap_text = text[prev_end:start]
        gap_len = start - prev_end

        should_merge = (
            label == prev_label and
            gap_len >= 0 and
            gap_len <= max_gap and
            gap_text.isspace()
        )

        if should_merge:
            merged[-1] = (prev_start, end, label)
        else:
            merged.append((start, end, label))

    return merged


def filter_short_suspicious_spans(text, spans):
    """
    Remove very short single-token predictions that are often noise.
    Keep all spans of length > 2 chars.
    """
    filtered = []
    for start, end, label in spans:
        span_text = text[start:end].strip()

        if len(span_text) <= 2:
            continue

        filtered.append((start, end, label))

    return filtered


def predict_document_spans(model, tokenizer, text, max_length=256, stride=64, device="cpu"):
    merged_tokens = aggregate_token_predictions(
        model=model,
        tokenizer=tokenizer,
        text=text,
        max_length=max_length,
        stride=stride,
        device=device,
    )
    spans = decode_spans_from_aggregated_tokens(merged_tokens)
    spans = clean_predicted_spans(text, spans)
    spans = merge_adjacent_same_type_spans(text, spans, max_gap=1)
    # spans = filter_short_suspicious_spans(text, spans)  # WARNING: Filter!
    return spans


def exact_span_f1(gold_docs, pred_map):
    gold = set()
    pred = set()

    for doc in gold_docs:
        doc_id = doc["doc_id"]
        for s in doc["spans"]:
            gold.add((doc_id, s["start"], s["end"], s["label"].upper()))

        for s, e, t in pred_map.get(doc_id, []):
            pred.add((doc_id, s, e, t))

    tp = len(gold & pred)
    fp = len(pred - gold)
    fn = len(gold - pred)

    precision = tp / (tp + fp) if tp + fp > 0 else 0.0
    recall = tp / (tp + fn) if tp + fn > 0 else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall > 0 else 0.0

    return {
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "tp": tp,
        "fp": fp,
        "fn": fn,
    }

### Validate

Predicted label distribution on validation

In [ ]:
pred_counter = Counter()

model.eval()
with torch.no_grad():
    for rec in valid_ds.records:
        input_ids = torch.tensor([rec.input_ids], device=device)
        attention_mask = torch.tensor([rec.attention_mask], device=device)

        logits = model(input_ids=input_ids, attention_mask=attention_mask).logits
        pred_ids = logits.argmax(-1).squeeze(0).cpu().tolist()

        for pred_id, (start, end) in zip(pred_ids, rec.offsets):
            if start == 0 and end == 0:
                continue
            pred_counter[ID2LABEL[pred_id]] += 1

print(pred_counter)

Counter({'O': 263982, 'I-SYMPTOM': 77650, 'I-DISEASE': 60921, 'I-PROCEDURE': 57222, 'B-SYMPTOM': 18174, 'B-PROCEDURE': 17104, 'B-DISEASE': 13300})


In [ ]:
rec = valid_ds.records[0]

model.eval()
with torch.no_grad():
    input_ids = torch.tensor([rec.input_ids], device=device)
    attention_mask = torch.tensor([rec.attention_mask], device=device)

    logits = model(input_ids=input_ids, attention_mask=attention_mask).logits
    pred_ids = logits.argmax(-1).squeeze(0).cpu().tolist()

tokens = tokenizer.convert_ids_to_tokens(rec.input_ids)
gold_labels = [ID2LABEL[x] if x != -100 else "IGN" for x in rec.labels]
pred_labels = [ID2LABEL[x] for x in pred_ids]

for tok, gold, pred, (s, e) in list(zip(tokens, gold_labels, pred_labels, rec.offsets))[:120]:
    print(f"{tok:20s} gold={gold:15s} pred={pred:15s} offset=({s},{e})")

[CLS]                gold=IGN             pred=O               offset=(0,0)
a                    gold=O               pred=O               offset=(0,1)
42                   gold=O               pred=O               offset=(2,4)
-                    gold=O               pred=O               offset=(4,5)
year                 gold=O               pred=O               offset=(5,9)
-                    gold=O               pred=O               offset=(9,10)
old                  gold=O               pred=O               offset=(10,13)
woman                gold=O               pred=O               offset=(14,19)
consulted            gold=O               pred=O               offset=(20,29)
for                  gold=O               pred=O               offset=(30,33)
a                    gold=O               pred=O               offset=(34,35)
history              gold=O               pred=O               offset=(36,43)
of                   gold=O               pred=O               offset=(44,4

Validation prediction check

In [ ]:
model.to(device)

doc = valid_docs[0]
pred_spans = predict_document_spans(
    model,
    tokenizer,
    doc["text"],
    max_length=MAX_LENGTH,
    stride=STRIDE,
    device=device,
)

print("doc_id:", doc["doc_id"])
print("\nGold spans:")
print([(s["start"], s["end"], s["label"]) for s in doc["spans"][:20]])

print("\nPredicted spans:")
print(pred_spans[:20])

doc_id: MultiClinNER-en-train-procedure-0569

Gold spans:
[(216, 240, 'PROCEDURE'), (339, 354, 'PROCEDURE'), (519, 558, 'PROCEDURE'), (589, 596, 'PROCEDURE'), (878, 888, 'PROCEDURE'), (936, 1031, 'PROCEDURE'), (1134, 1143, 'PROCEDURE'), (1163, 1186, 'PROCEDURE'), (1300, 1326, 'PROCEDURE'), (1544, 1585, 'PROCEDURE'), (1811, 1823, 'PROCEDURE'), (1881, 1890, 'PROCEDURE')]

Predicted spans:
[(47, 55, 'SYMPTOM'), (92, 108, 'SYMPTOM'), (121, 129, 'DISEASE'), (131, 161, 'DISEASE'), (162, 163, 'SYMPTOM'), (163, 166, 'DISEASE'), (166, 169, 'SYMPTOM'), (169, 170, 'DISEASE'), (188, 196, 'SYMPTOM'), (200, 210, 'DISEASE'), (216, 240, 'PROCEDURE'), (250, 265, 'SYMPTOM'), (272, 280, 'SYMPTOM'), (281, 282, 'PROCEDURE'), (282, 294, 'SYMPTOM'), (294, 300, 'PROCEDURE'), (305, 337, 'DISEASE'), (339, 354, 'PROCEDURE'), (365, 421, 'SYMPTOM'), (427, 433, 'DISEASE')]


In [ ]:
doc = valid_docs[0]

pred_spans = predict_document_spans(
    model,
    tokenizer,
    doc["text"],
    max_length=MAX_LENGTH,
    stride=STRIDE,
    device=device,
)

print("doc_id:", doc["doc_id"])
print("Num gold:", len(doc["spans"]))
print("Num pred:", len(pred_spans))

print("\nGold spans:")
for x in doc["spans"][:20]:
    print((x["start"], x["end"], x["label"], repr(doc["text"][x["start"]:x["end"]])))

print("\nPredicted spans:")
for x in pred_spans[:40]:
    s, e, t = x
    print((s, e, t, repr(doc["text"][s:e])))

doc_id: MultiClinNER-en-train-procedure-0569
Num gold: 12
Num pred: 64

Gold spans:
(216, 240, 'PROCEDURE', "'neurological examination'")
(339, 354, 'PROCEDURE', "'Cranial CT scan'")
(519, 558, 'PROCEDURE', "'Magnetic resonance imaging of the brain'")
(589, 596, 'PROCEDURE', "'CT scan'")
(878, 888, 'PROCEDURE', "'Gadolinium'")
(936, 1031, 'PROCEDURE', "'tumour was excised almost completely by means of a craniectomy of the right posterior hemifossa'")
(1134, 1143, 'PROCEDURE', "'brain MRI'")
(1163, 1186, 'PROCEDURE', "'resection of the tumour'")
(1300, 1326, 'PROCEDURE', "'anatomo-pathological study'")
(1544, 1585, 'PROCEDURE', "'focal radiotherapy in the posterior fossa'")
(1811, 1823, 'PROCEDURE', "'intervention'")
(1881, 1890, 'PROCEDURE', "'brain MRI'")

Predicted spans:
(47, 55, 'SYMPTOM', "'headache'")
(92, 108, 'SYMPTOM', "'gait instability'")
(121, 129, 'DISEASE', "'migraine'")
(131, 161, 'DISEASE', '\'peripheral facial paralysis "a\'')
(162, 163, 'SYMPTOM', "'f'")
(163, 166, 'D

Full valication metrics

In [ ]:
pred_map = {}
for i, doc in enumerate(valid_docs):
    if i % 20 == 0:
        print(f"{i}/{len(valid_docs)}")
    pred_map[doc["doc_id"]] = predict_document_spans(
        model,
        tokenizer,
        doc["text"],
        max_length=MAX_LENGTH,
        stride=STRIDE,
        device=device,
    )

metrics = exact_span_f1(valid_docs, pred_map)
print(json.dumps(metrics, indent=2))

with open(OUTPUT_DIR / "validation_metrics.json", "w", encoding="utf-8") as f:
    json.dump(metrics, f, indent=2)

0/377
20/377
40/377
60/377
80/377
100/377
120/377
140/377
160/377
180/377
200/377
220/377
240/377
260/377
280/377
300/377
320/377
340/377
360/377
{
  "precision": 0.13699043232721989,
  "recall": 0.5942442621001308,
  "f1": 0.22265294301118388,
  "tp": 4997,
  "fp": 31480,
  "fn": 3412
}


### Save BRAT `.ann` predictions for validation or test text files

Use this for any folder containing `.txt` files.

In [ ]:
def write_brat_ann(output_path: Path, text: str, spans):
    lines = []
    for i, (start, end, label) in enumerate(sorted(spans, key=lambda x: (x[0], x[1], x[2])), start=1):
        mention = text[start:end].replace("\n", " ")
        lines.append(f"T{i}\t{label} {start} {end}\t{mention}")
    output_path.write_text("\n".join(lines) + ("\n" if lines else ""), encoding="utf-8")


def predict_folder_to_ann(txt_dir: Path, output_ann_dir: Path):
    output_ann_dir.mkdir(parents=True, exist_ok=True)

    txt_files = sorted(txt_dir.glob("*.txt"))
    print("Files:", len(txt_files))

    for i, txt_path in enumerate(txt_files):
        if i % 20 == 0:
            print(f"{i}/{len(txt_files)}")

        text = txt_path.read_text(encoding="utf-8")
        spans = predict_document_spans(
            model,
            tokenizer,
            text,
            max_length=MAX_LENGTH,
            stride=STRIDE,
            device=device,
        )
        write_brat_ann(output_ann_dir / f"{txt_path.stem}.ann", text, spans)

    print("Done:", output_ann_dir)

In [ ]:
PREDICT_TXT_DIR = Path("/content/drive/MyDrive/Colab Notebooks/some_folder_with_txt_only")
PREDICT_ANN_DIR = Path("/content/drive/MyDrive/Colab Notebooks/predicted_ann")

predict_folder_to_ann(PREDICT_TXT_DIR, PREDICT_ANN_DIR)

Files: 0
Done: /content/drive/MyDrive/Colab Notebooks/predicted_ann


In [ ]:
print(PREDICT_TXT_DIR)

/content/drive/MyDrive/Colab Notebooks/some_folder_with_txt_only
